In [166]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


In [167]:
dataset = pd.read_csv('Cleaned_UNSW-NB15.csv')

In [168]:
dataset.head()

,sport,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,1390.0,53.0,udp,CON,0.001055,132,164,31,29,0,...,0.0,3,7,1,3,1,1,1,Normal,0
1,33661.0,1024.0,udp,CON,0.036133,528,304,31,29,0,...,0.0,2,4,2,3,1,1,2,Normal,0
2,1464.0,53.0,udp,CON,0.001119,146,178,31,29,0,...,0.0,12,8,1,2,2,1,1,Normal,0
3,3593.0,53.0,udp,CON,0.001209,132,164,31,29,0,...,0.0,6,9,1,1,1,1,1,Normal,0
4,49664.0,53.0,udp,CON,0.001169,146,178,31,29,0,...,0.0,7,9,1,1,1,1,1,Normal,0


In [169]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2058991 entries, 0 to 2058990
Data columns (total 47 columns):
 #   Column            Dtype  
---  ------            -----  
 0   sport             float64
 1   dsport            float64
 2   proto             str    
 3   state             str    
 4   dur               float64
 5   sbytes            int64  
 6   dbytes            int64  
 7   sttl              int64  
 8   dttl              int64  
 9   sloss             int64  
 10  dloss             int64  
 11  service           str    
 12  Sload             float64
 13  Dload             float64
 14  Spkts             int64  
 15  Dpkts             int64  
 16  swin              int64  
 17  dwin              int64  
 18  stcpb             int64  
 19  dtcpb             int64  
 20  smeansz           int64  
 21  dmeansz           int64  
 22  trans_depth       int64  
 23  res_bdy_len       int64  
 24  Sjit              float64
 25  Djit              float64
 26  Stime             str    

In [170]:
dataset.columns

Index(['sport', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl',
       'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts',
       'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth',
       'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt',
       'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
       'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src',
       'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm',
       'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label'],
      dtype='str')

In [171]:
dataset['tcp_timing_mean'] = dataset[
    ['tcprtt', 'synack', 'ackdat']
].mean(axis=1)

In [172]:
dataset[["tcprtt", "synack", "ackdat", "tcp_timing_mean", "Label"]].corr()

,tcprtt,synack,ackdat,tcp_timing_mean,Label
tcprtt,1.000000,0.930215,0.917777,1.000000,0.283248
synack,0.930215,1.000000,0.707988,0.930215,0.246365
ackdat,0.917777,0.707988,1.000000,0.917777,0.278478
tcp_timing_mean,1.000000,0.930215,0.917777,1.000000,0.283248
Label,0.283248,0.246365,0.278478,0.283248,1.000000


I created tcp_timing_mean as an aggregated feature to combine the information from tcprtt, synack, and ackdat. Since the new feature is highly correlated with the original features, I can retain the aggregated feature and remove the three original features to reduce redundancy

In [173]:
dataset = dataset.drop(columns=["tcprtt", "synack", "ackdat"])

In [174]:
x = dataset.drop(columns=["Label", "attack_cat"])
y= dataset["Label"]

In [175]:
x.dtypes.value_counts()

int64      26
float64    12
str         5
Name: count, dtype: int64

In [176]:
categorical_features = x.select_dtypes(include=['str']).columns

In [177]:
print(categorical_features)

Index(['proto', 'state', 'service', 'Stime', 'Ltime'], dtype='str')


In [178]:
for col in categorical_features:
  display( dataset[col].unique())

<ArrowStringArray>
[     'udp',      'arp',      'tcp',     'ospf',     'icmp',     'igmp',
     'sctp',      'udt',      'sep',   'sun-nd',
 ...
     'crtp',     'isis',    'crudp', 'sccopmce',      'sps',     'pipe',
     'iplt',     'unas',       'fc',       'ib']
Length: 135, dtype: str

<ArrowStringArray>
['CON', 'INT', 'FIN', 'URH', 'REQ', 'ECO', 'RST', 'CLO', 'TXD', 'URN',  'no',
 'ACC', 'PAR', 'MAS', 'TST', 'ECR']
Length: 16, dtype: str

<ArrowStringArray>
[     'dns',     'none',     'http',     'smtp', 'ftp-data',      'ftp',
      'ssh',     'pop3',     'snmp',      'ssl',      'irc',   'radius',
     'dhcp']
Length: 13, dtype: str

<ArrowStringArray>
['1970-01-01 00:00:01.421927414', '1970-01-01 00:00:01.421927415',
 '1970-01-01 00:00:01.421927416', '1970-01-01 00:00:01.421927417',
 '1970-01-01 00:00:01.421927418', '1970-01-01 00:00:01.421927419',
 '1970-01-01 00:00:01.421927420', '1970-01-01 00:00:01.421927421',
 '1970-01-01 00:00:01.421927422', '1970-01-01 00:00:01.421927423',
 ...
 '1970-01-01 00:00:01.424262059', '1970-01-01 00:00:01.424262060',
 '1970-01-01 00:00:01.424262061', '1970-01-01 00:00:01.424262062',
 '1970-01-01 00:00:01.424262063', '1970-01-01 00:00:01.424262064',
 '1970-01-01 00:00:01.424262065', '1970-01-01 00:00:01.424262066',
 '1970-01-01 00:00:01.424262067', '1970-01-01 00:00:01.424262068']
Length: 85348, dtype: str

<ArrowStringArray>
['1970-01-01 00:00:01.421927414', '1970-01-01 00:00:01.421927415',
 '1970-01-01 00:00:01.421927416', '1970-01-01 00:00:01.421927417',
 '1970-01-01 00:00:01.421927418', '1970-01-01 00:00:01.421927419',
 '1970-01-01 00:00:01.421927420', '1970-01-01 00:00:01.421927421',
 '1970-01-01 00:00:01.421927422', '1970-01-01 00:00:01.421927423',
 ...
 '1970-01-01 00:00:01.424262060', '1970-01-01 00:00:01.424262061',
 '1970-01-01 00:00:01.424262062', '1970-01-01 00:00:01.424262063',
 '1970-01-01 00:00:01.424262064', '1970-01-01 00:00:01.424262065',
 '1970-01-01 00:00:01.424262066', '1970-01-01 00:00:01.424262067',
 '1970-01-01 00:00:01.424262068', '1970-01-01 00:00:01.424262069']
Length: 85361, dtype: str

In [179]:
x["Stime"] = pd.to_datetime(x["Stime"])
x["Ltime"] = pd.to_datetime(x["Ltime"])

In [180]:
print(x[["Stime", "Ltime"]].corrwith(y))

Stime    0.15892
Ltime    0.15892
dtype: float64


In [181]:
x.drop(columns=["Stime", "Ltime"], inplace=True)

In [182]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [183]:
categorical_features = x_train.select_dtypes(include=['str']).columns

In [190]:
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output =False)
train_encoded = encoder.fit_transform(x_train[categorical_features])
test_encoded = encoder.transform(x_test[categorical_features])


In [191]:
encoded_columns = encoder.get_feature_names_out(categorical_features)

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_columns,
    index=x_train.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_columns,
    index=x_test.index
)

In [194]:
x_train = pd.concat(
    [
        x_train.drop(columns=categorical_features),
        train_encoded_df
    ],
    axis=1
)

x_test = pd.concat(
    [
        x_test.drop(columns=categorical_features),
        test_encoded_df
    ],
    axis=1
)

In [195]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(x_train)
test_scaled = scaler.transform(x_test)

In [198]:
numeric_features = x_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = x_train.select_dtypes(
    include=["str"]
).columns.tolist()

In [202]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", scaler, numeric_features),
        ("cat", encoder, categorical_features)
    ]
)

In [204]:
from sklearn.pipeline import Pipeline

preprocessing_pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

In [205]:
import joblib

joblib.dump(
    preprocessing_pipeline,
    "preprocessing_pipeline.joblib"
)

['preprocessing_pipeline.joblib']